# CASE 02 Data Quality Check

원본 엑셀은 수정하지 않는다. 공식 부표의 교차 대사만 수행한다.

In [1]:
from pathlib import Path
import sys

CASE_NAME = "02_youth_migration_dynamics"
RAW_NAME = "2025_domestic_migration_statistics.xlsx"
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "cases" / CASE_NAME,
]
CASE_DIR = next(
    (path.resolve() for path in candidates if (path / "data" / "raw" / RAW_NAME).exists()),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(
        f"{RAW_NAME}를 찾지 못했습니다. 저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )
sys.path.insert(0, str(CASE_DIR / "src"))

from constants import RAW_FILE_NAME
from data_preparation import load_and_prepare
from data_quality import run_quality_checks
from kpi_segmentation import run_kpi_segmentation
from parse_official_tables import verify_source_file
from statistical_analysis import run_statistical_analysis

RAW = CASE_DIR / "data" / "raw" / RAW_FILE_NAME
source = verify_source_file(RAW)
prepared = load_and_prepare(RAW)
tables = prepared.tables
print(source["sha256"])
print("stale sheet present:", tables.workbook["has_stale_monthly_sheet"])


FE066C40AAE0AE5C34C67947B404DC8943405C9BF0A7952DA09B0CF26D901E9D
stale sheet present: True


In [2]:
quality = run_quality_checks(tables, prepared)
quality

,check_name,passed,actual,expected,note
0,headline_total_movers_2025,True,6117784,6117784,보도 헤드라인과 표 1의 2025 총이동자 수
1,intra_plus_inter_equals_total,True,6117784,6117784,
2,male_plus_female_equals_total,True,6117784,6117784,
3,sheet3_national_in_equals_sheet1,True,6117784,6117784,
4,national_net_is_zero,True,0,0,
5,sheet5_total_matches_sheet3_net,True,0,0,
6,age_bands_sum_to_sido_total_net,True,0,0,0-4부터 80+까지 합이 계와 같은지
7,male_female_net_sum_to_all,True,0,0,
8,od_row_is_destination,True,438327,438327,"행=전입지, 열=전출지. 서울 행의 전국-대각 = 시도간 전입"
9,od_net_equals_mover_difference,True,-41199,-41199,


In [3]:
assert quality['passed'].all(), quality.loc[~quality['passed']]

## 잠긴 해석

- 청년 = 20-24 + 25-29 + 30-34 + 35-39세. e-지방지표 19-39와 직접 대사하지 않는다.
- OD 행렬은 행=전입지, 열=전출지다. 서울 행의 전국-대각 = 시도간 전입과 일치한다.
- `8.월별` 시트는 2009-2011 잔여 표로 보이며 분석에서 제외한다.